In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [2]:
train = pd.read_csv("titanic/train.csv", index_col="PassengerId")
test = pd.read_csv("titanic/test.csv", index_col="PassengerId")

In [3]:
numerical_cols = train.select_dtypes(include="number").columns
categorical_cols = train.select_dtypes(include=["str", "category"]).columns

In [4]:
#Cabin ist etwas nutzlos eigentlich
for df in [train, test]:
    df['Deck'] = df['Cabin'].str[0]
    df['Deck'] = df['Deck'].fillna('0')

In [5]:
#Zu NA Werten. In der Baseline Logistic Regression werde ich es einfach imputen. Datenlecks werden nicht auftreten. Boosting kommt da schon ohne klar

In [6]:
train.info()

<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 1 to 891
Data columns (total 12 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   Survived  891 non-null    int64  
 1   Pclass    891 non-null    int64  
 2   Name      891 non-null    str    
 3   Sex       891 non-null    str    
 4   Age       714 non-null    float64
 5   SibSp     891 non-null    int64  
 6   Parch     891 non-null    int64  
 7   Ticket    891 non-null    str    
 8   Fare      891 non-null    float64
 9   Cabin     204 non-null    str    
 10  Embarked  889 non-null    str    
 11  Deck      891 non-null    str    
dtypes: float64(2), int64(4), str(6)
memory usage: 83.7 KB


In [7]:
train[(train["Sex"] == "male") & (train["Age"] < 18)]

,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,Deck
PassengerId,,,,,,,,,,,,
8,0,3,"Palsson, Master. Gosta Leonard",male,2.00,3,1,349909,21.0750,NaN,S,0
17,0,3,"Rice, Master. Eugene",male,2.00,4,1,382652,29.1250,NaN,Q,0
51,0,3,"Panula, Master. Juha Niilo",male,7.00,4,1,3101295,39.6875,NaN,S,0
60,0,3,"Goodwin, Master. William Frederick",male,11.00,5,2,CA 2144,46.9000,NaN,S,0
64,0,3,"Skoog, Master. Harald",male,4.00,3,2,347088,27.9000,NaN,S,0
79,1,2,"Caldwell, Master. Alden Gates",male,0.83,0,2,248738,29.0000,NaN,S,0
87,0,3,"Ford, Mr. William Neal",male,16.00,1,3,W./C. 6608,34.3750,NaN,S,0
126,1,3,"Nicola-Yarred, Master. Elias",male,12.00,1,0,2651,11.2417,NaN,C,0
139,0,3,"Osen, Mr. Olaf Elon",male,16.00,0,0,7534,9.2167,NaN,S,0


In [8]:
train[train["Name"].str.contains("Master.", na=False)]

,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,Deck
PassengerId,,,,,,,,,,,,
8,0,3,"Palsson, Master. Gosta Leonard",male,2.00,3,1,349909,21.0750,NaN,S,0
17,0,3,"Rice, Master. Eugene",male,2.00,4,1,382652,29.1250,NaN,Q,0
51,0,3,"Panula, Master. Juha Niilo",male,7.00,4,1,3101295,39.6875,NaN,S,0
60,0,3,"Goodwin, Master. William Frederick",male,11.00,5,2,CA 2144,46.9000,NaN,S,0
64,0,3,"Skoog, Master. Harald",male,4.00,3,2,347088,27.9000,NaN,S,0
66,1,3,"Moubarek, Master. Gerios",male,NaN,1,1,2661,15.2458,NaN,C,0
79,1,2,"Caldwell, Master. Alden Gates",male,0.83,0,2,248738,29.0000,NaN,S,0
126,1,3,"Nicola-Yarred, Master. Elias",male,12.00,1,0,2651,11.2417,NaN,C,0
160,0,3,"Sage, Master. Thomas Henry",male,NaN,8,2,CA. 2343,69.5500,NaN,S,0


In [9]:
#Titel
for df in [train, test]:
    df['Title'] = df['Name'].str.extract(r',\s*([^\.]+)\.')
    keep = ['Mr', 'Mrs', 'Miss', 'Master', 'Dr']
    df['Title'] = df['Title'].where(df['Title'].isin(keep), other='Rare')

    df['FamilySize'] = df['SibSp'] + df['Parch'] + 1  # +1 für die Person selbst
    df['IsAlone'] = (df['FamilySize'] == 1).astype(int)

In [10]:
def family_category(size):
    if size == 1:
        return 'Alone'
    elif size <= 4:
        return 'Small'
    else:
        return 'Large'
for df in [train, test]:
    df['FamilyCategory'] = df['FamilySize'].apply(family_category)

In [11]:
train.groupby('FamilySize')['Survived'].agg(['mean', 'count'])

,mean,count
FamilySize,,
1,0.303538,537
2,0.552795,161
3,0.578431,102
4,0.724138,29
5,0.200000,15
6,0.136364,22
7,0.333333,12
8,0.000000,6
11,0.000000,7


In [12]:
train['Title'].value_counts()

Title
Mr        517
Miss      182
Mrs       125
Master     40
Rare       20
Dr          7
Name: count, dtype: int64

In [13]:
for df in [train, test]:
    df["Embarked"] = df["Embarked"].fillna("S")

    df["Log_Fare"] = np.log1p(df["Fare"])

In [14]:
#Geschätzte Age-Werte: Ich nehme an, die Schätzungen sind korrekt
for df in [train, test]:
    df['Age'] = df.groupby('Title')['Age'].transform(
        lambda x: x.fillna(x.median())
    )
    #Kind-Flag
    df['IsChild'] = (df['Age'].dropna() < 18).astype(int)

train.Age = np.floor(train.Age).astype("Int64")
test.Age = np.floor(test.Age).astype("Int64")

In [15]:
combined_tickets = pd.concat([train['Ticket'], test['Ticket']])
ticket_counts_combined = combined_tickets.value_counts()

train['TicketFrequency'] = train['Ticket'].map(ticket_counts_combined)
test['TicketFrequency'] = test['Ticket'].map(ticket_counts_combined)

In [16]:
train = train.drop(columns=["Parch", "SibSp", "Name", "Cabin", "Ticket"])
test = test.drop(columns=["Parch", "SibSp", "Name", "Cabin", "Ticket"])

In [18]:
train.Deck.value_counts()

Deck
0    687
C     59
B     47
D     33
E     32
A     15
F     13
G      4
T      1
Name: count, dtype: int64

In [17]:
train.to_csv("train.csv", index=True)
test.to_csv("test.csv", index=True)